In [ ]:
# PART 1: Read data

import mne
import numpy as np
import scipy.io
from tqdm import tqdm
import pickle
mne.set_log_level("ERROR")

# EMOEEG DATASET
sfreq = 200
n_sub = 1
volt_factor = 1e-6
sub_start = 0
sub_end = 
sub_end = sub_end + 1
channel_names = [
'FP1', 'FPZ', 'FP2', 'AF7', 'AF3','AF4','AF8', 'F7', 'F5','F3','F1','FZ', 'F2', 'F4', 'F6', 'F8',
'FT7', 'FC5', 'FC3', 'FC1','FCZ','FC2','FC4', 'FC6', 'FT8', 'T7','C5', 'C3', 'C1', 'CZ', 'C2', 'C4', 'C6', 'T8',
'TP7', 'CP5', 'CP3', 'CP1','CPZ','CP2', 'CP4','CP6', 'TP8', 'P7','P5', 'P3', 'P1', 'PZ','P2', 'P4', 'P6', 'P8',
'PO7', 'PO3','POZ', 'PO4','PO8', 'O1','OZ','O2', 'F9', 'F10', 'TP9', 'TP10'
]


raws = []
for sub in tqdm(range(sub_start, sub_end), desc=f'Reading sub:'):
    raws_sub = []
    valid_trials_sub = []
    data_path = f'Z:/xuxin/new_pre/trials/sub{sub:02}.pkl'
    with open(data_path, 'rb') as f:
        eeg_data_trials = pickle.load(f)
        ima_data = []
        vid_data = []
        for idx, channel_data in enumerate(eeg_data_trials):
            if(idx % 2 == 0):
                ima_data.append(channel_data)
            if(idx % 2 == 1):
                vid_data.append(channel_data)
        info = mne.create_info(ch_names=channel_names, sfreq=sfreq, ch_types='eeg')
        for trial_id in range(42):
            # eeg_trial_i = eeg_data_trials[str(trial_id+1)]
            eeg_trial_i = []
            for cha in range(64):
                if(trial_id < 21):
                    eeg_trial_i.append(ima_data[cha][trial_id])
                if(trial_id >= 21):
                    eeg_trial_i.append(vid_data[cha][trial_id-21])
            eeg_trial_i = np.array(eeg_trial_i)
            print(eeg_trial_i.shape)
            try:
                raw_i = mne.io.RawArray(eeg_trial_i * volt_factor, info)
                raws_sub.append(raw_i)
            except:
                print(f'ERROR Sub {sub+1} Trial {trial_id}')
                raws_sub.append(None)
                continue
        raws.append(raws_sub)
    


Reading sub::   0%|          | 0/1 [00:00<?, ?it/s]

(64, 11956)
(64, 11919)
(64, 9609)
(64, 10928)
(64, 14112)
(64, 13696)
(64, 5877)
(64, 9832)
(64, 8840)
(64, 15284)
(64, 15068)
(64, 18258)
(64, 15045)
(64, 21213)
(64, 13770)
(64, 12299)
(64, 16904)
(64, 21784)
(64, 14437)
(64, 17560)
(64, 19070)
(64, 19461)
(64, 18090)
(64, 14383)
(64, 13226)
(64, 19919)
(64, 23119)
(64, 21079)


Reading sub:: 100%|██████████| 1/1 [00:12<00:00, 12.32s/it]

(64, 22192)
(64, 20723)
(64, 14604)
(64, 50963)
(64, 16559)
(64, 12968)
(64, 22086)
(64, 11405)
(64, 16117)
(64, 14338)
(64, 21700)
(64, 24150)
(64, 22962)
(64, 21401)


In [2]:
raws[0][0].plot()

In [ ]:
# Select eeg channels
eeg_channel_names = [
    "FP1", "FPZ", "FP2", 
    "AF3", "AF4", 
    "F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8", 
    "FT7", "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6", "FT8", 
    "T7", "C5", "C3", "C1", "CZ", "C2", "C4", "C6", "T8", 
    "TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6", "TP8", 
    "P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8", 
    "PO7", "PO3", "POZ", "PO4", "PO8", 
    "O1", "OZ", "O2",
]
eeg_selected = []
debug = 1
for sub in tqdm(range(sub_start, sub_end), desc='Selecting Channel'):
    eeg_selected_sub = []
    raw_sub = raws[sub]
    for trial_i in raw_sub:
        
        if(debug):
            raw_selected = trial_i.copy().pick_channels(eeg_channel_names)
            eeg_selected_sub.append(raw_selected)
        else:
            try:
                raw_selected = trial_i.copy().pick_channels(eeg_channel_names)
                eeg_selected_sub.append(raw_selected)
            except:
                eeg_selected_sub.append(None)
    eeg_selected.append(eeg_selected_sub)

Selecting Channel:   0%|          | 0/1 [00:00<?, ?it/s]


IndexError: list index out of range

In [ ]:
eeg_selected[0][0].plot()

In [ ]:
# Filter & downsample eeg data
new_rate = 125
eeg_filted = []
for sub in tqdm(range(sub_start, sub_end), desc='Filtering data'):
    eeg_filted_sub = []
    eeg_selected_sub = eeg_selected[sub]
    for trial_i in eeg_selected_sub:
        try:
            trial_i_downsampled = trial_i.resample(sfreq=new_rate)
            trial_i_filted = trial_i_downsampled.filter(1, 47, fir_design='firwin')
            eeg_filted_sub.append(trial_i_filted)
        except:
            eeg_filted_sub.append(None)
    eeg_filted.append(eeg_filted_sub)

Filtering data: 100%|██████████| 1/1 [00:02<00:00,  2.28s/it]


In [ ]:
# 1st bad channel interpolation eeg data
def detect_bad_channels(raw, thresholds):
    data = raw.get_data(picks='eeg')  # 获取 EEG 数据，形状为 (n_channels, n_times)
    sfreq = raw.info['sfreq']         # 采样频率
    total_samples = data.shape[1]     # 总采样点数
    bad_channels = set()              # 使用集合存储坏道以避免重复
    for a, b in thresholds:
        for ch_idx, ch_data in enumerate(data):
            median = np.median(np.abs(ch_data))
            high_values = np.abs(ch_data) > (a * median)
            high_ratio = np.sum(high_values) / total_samples
            if high_ratio > b:
                bad_channels.add(raw.info['ch_names'][ch_idx])
    return list(bad_channels)

thresholds = [
    (3, 0.4),
    (30, 0.01)
]


montage_file_path = 'Z:/qingzhu/EEG_raw/SEED-VII/src/channel_62_pos.locs'
montage = mne.channels.read_custom_montage(montage_file_path)


eeg_interpolated = []
for sub in tqdm(range(sub_start, sub_end), desc='1st bad channel inter'):
    eeg_interpolated_sub = []
    eeg_filted_sub = eeg_filted[sub]
    for trial_i in eeg_filted_sub:
    
        try:
            trial_i_interploated = trial_i.copy()
            bad_channels = detect_bad_channels(trial_i, thresholds)
            print(f"Detected bad channels: {bad_channels}")
            trial_i_interploated.info['bads'] = bad_channels
            trial_i_interploated.set_montage(montage)
            trial_i_interploated.interpolate_bads(reset_bads=True, exclude=['FP1', 'FP2', 'F7', 'F8', 'AF7', 'AF8'])
            eeg_interpolated_sub.append(trial_i_interploated)
        except:
            eeg_interpolated_sub.append(None)
    eeg_interpolated.append(eeg_interpolated_sub)

1st bad channel inter:   0%|          | 0/1 [00:00<?, ?it/s]

Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: ['FP2', 'FP1']
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: ['FP2', 'FP1']
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad channels: []
Detected bad c

1st bad channel inter: 100%|██████████| 1/1 [00:01<00:00,  1.16s/it]

Detected bad channels: []
Detected bad channels: []


In [ ]:
# ICA denoise
from mne_icalabel import label_components
from mne.preprocessing import ICA
def ICA_denoise(eeg):
    eeg = eeg.set_eeg_reference("average")
    n_components = np.floor(len(eeg.ch_names) * 0.78).astype(int)
    ica = ICA(n_components=n_components, max_iter="auto", random_state=42, method='infomax', fit_params=dict(extended=True))
    # print(f'num_of_nan: {np.isnan(eeg.get_data()).sum()}')
    ica.fit(eeg)
    eeg.load_data()
    ic_labels = label_components(eeg, ica, method="iclabel")
    labels = ic_labels["labels"]
    exclude_idx = [
        idx for idx, label in enumerate(labels) if label not in ["brain", "other"]
    ]
    eeg_ica = eeg.copy()
    ica.apply(eeg_ica, exclude=exclude_idx)
    return eeg_ica


In [ ]:

eeg_denoised = []
for sub in tqdm(range(sub_start, sub_end), desc='ICA denoising'):
    eeg_denoised_sub = []
    eeg_interpolated_sub = eeg_interpolated[sub]
    for i, trial_i in enumerate(eeg_interpolated_sub):
        try:
            trial_i_ica = trial_i.copy()
            trial_i_ica = ICA_denoise(trial_i_ica)
            eeg_denoised_sub.append(trial_i_ica)
            print(f"Success ICA on sub {sub} trial {i}")
        except:
            print(f"Failed ICA on sub {sub} trial {i}")
            eeg_denoised_sub.append(None)
    eeg_denoised.append(eeg_denoised_sub)

ICA denoising:   0%|          | 0/1 [00:00<?, ?it/s]

Success ICA on sub 0 trial 0
Success ICA on sub 0 trial 1
Success ICA on sub 0 trial 2
Success ICA on sub 0 trial 3
Success ICA on sub 0 trial 4
Failed ICA on sub 0 trial 5
Success ICA on sub 0 trial 6


: 

: 

In [ ]:
# 2nd bad channel interploation
eeg_interpolated_2nd = []
for eeg_denoised_sub in eeg_denoised:
    eeg_interpolated_2nd_sub = []
    for trial_i in eeg_denoised_sub:
        try:
            trial_i_interpolated = trial_i.copy()
            bad_channels = detect_bad_channels(trial_i_interpolated, thresholds)
            print(f"Detected bad channels: {bad_channels}")
            trial_i_interpolated.info['bads'] = bad_channels
            trial_i_interploated.set_montage(montage)
            trial_i_interpolated.interpolate_bads(reset_bads=True)
            trial_i_interpolated.set_eeg_reference("average")
            eeg_interpolated_2nd_sub.append(trial_i_interpolated)
        except:
            eeg_interpolated_2nd_sub.append(None)
    eeg_interpolated_2nd.append(eeg_interpolated_2nd_sub)

In [ ]:
# Save data
import os
save_dir = 'Z:/qingzhu/AutoICA_Processed_EEG/EMOEEG'
for sub_id in range(sub_start, sub_end):
    eeg_sub = eeg_interpolated_2nd[sub_id]
    for trial_id, trial_data in enumerate(eeg_sub):
        sub_dir = os.path.join(save_dir, f'sub_{sub_id}')
        if not os.path.exists(sub_dir):  # 如果子路径不存在，创建它
            os.makedirs(sub_dir)
        try:
            data_dir = os.path.join(sub_dir, f'eeg_sub_{sub_id}_trial_{trial_id}')
            np.save(data_dir, trial_data.get_data())
        except:
            continue

Channels marked as bad:
none
